# Оценка моделей spellcheck

Сравнение **SAGE-FREDT5-Large**, **Qwen3.5-0.8B (base)** и **Qwen3.5-0.8B (fine-tuned)** на тестовой выборке [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE).

Метрики: RUSpellEval и ERRANT через библиотеку [SAGE](https://github.com/ai-forever/sage).

## 1. Окружение

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")


## 2. Тестовый датасет RU_SPELLCHECK_DEVICE

In [ ]:
from datasets import load_dataset

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
test_df = load_dataset(DATASET_NAME, split="test").to_pandas().dropna()

# typed — текст с опечатками; original — эталонная правка
sources = test_df["typed"].tolist()
corrections = test_df["original"].tolist()

print(f"Loaded {DATASET_NAME} (test): {len(test_df)} rows")
print(f"Columns: {list(test_df.columns)}")
print("\nExample:")
print(f"  typed:    {sources[0]}")
print(f"  original: {corrections[0]}")


## 3. MultidomainGold (подмножество literature)

Бенчмарк [`ai-forever/spellcheck_benchmark`](https://huggingface.co/datasets/ai-forever/spellcheck_benchmark) используется для анализа ошибок в домене literature.

In [ ]:
from datasets import concatenate_datasets, load_dataset

BENCHMARK_REPO = "ai-forever/spellcheck_benchmark"
MD_BASE = f"https://huggingface.co/datasets/{BENCHMARK_REPO}/resolve/main/data/MultidomainGold"

multidomain_gold = load_dataset(
    "json",
    data_files={
        "train": f"{MD_BASE}/train.json",
        "test": f"{MD_BASE}/test.json",
    },
)

md_df = concatenate_datasets(
    [multidomain_gold["train"], multidomain_gold["test"]]
).to_pandas()

literature_df = md_df[md_df["domain"] == "literature"]
literature_with_errors = literature_df[
    literature_df["source"] != literature_df["correction"]
]

print(f"MultidomainGold total: {len(md_df)} rows")
print(f"Domains: {sorted(md_df['domain'].unique())}")
print(f"Literature with errors: {len(literature_with_errors)} rows")


### 3.1 Распределение по domain

In [ ]:
import matplotlib.pyplot as plt

domain_counts = md_df["domain"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(
    domain_counts.values,
    labels=domain_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)
ax.set_title("MultidomainGold: распределение по domain")
plt.tight_layout()
plt.show()

print(domain_counts)


### 3.2 Примеры посимвольных различий (literature)

In [ ]:
import difflib


def char_diff_lines(source: str, correction: str) -> list[str]:
    """Позиции и символы, где source и correction расходятся."""
    sm = difflib.SequenceMatcher(None, source, correction)
    lines = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        if tag == "replace":
            for offset in range(max(i2 - i1, j2 - j1)):
                pos = i1 + offset
                sc = source[i1 + offset] if i1 + offset < i2 else None
                cc = correction[j1 + offset] if j1 + offset < j2 else None
                lines.append(f"  [{pos}] source={sc!r} → correction={cc!r}")
        elif tag == "delete":
            for pos in range(i1, i2):
                lines.append(f"  [{pos}] source={source[pos]!r} → correction=— (удалено)")
        elif tag == "insert":
            for j in range(j1, j2):
                lines.append(f"  [{i1}] source=— → correction={correction[j]!r} (вставка)")
    return lines


examples = literature_with_errors.head(10)
print(f"Показано {len(examples)} примеров\n")

for i, row in enumerate(examples.itertuples(), start=1):
    print(f"--- Пример {i} ---")
    print(f"Оригинал:\n{row.correction}\n")
    print(f"С опечатками:\n{row.source}\n")
    diffs = char_diff_lines(row.source, row.correction)
    print("Различия:")
    print("\n".join(diffs) if diffs else "  (нет различий)")
    print()


## 4. Установка SAGE и функции метрик

In [ ]:
!git clone https://github.com/ai-forever/sage.git
!pip install /content/sage
!pip install -e "/content/sage[errant]"
!python -m spacy download ru_core_news_lg


In [ ]:
import re
import sys
from pathlib import Path
from typing import Iterable

import difflib


def tokenize_spellcheck(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def extract_edits(source: str, target: str) -> dict[tuple[int, int], tuple[str, ...]]:
    src_tokens = tokenize_spellcheck(source)
    tgt_tokens = tokenize_spellcheck(target)
    edits: dict[tuple[int, int], tuple[str, ...]] = {}
    matcher = difflib.SequenceMatcher(None, src_tokens, tgt_tokens)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        replacement = tuple(tgt_tokens[j1:j2])
        if src_tokens[i1:i2] != list(replacement):
            edits[(i1, i2)] = replacement
    return edits


def extract_edits_corpus(
    sources: Iterable[str], targets: Iterable[str]
) -> dict[tuple[int, int, int], tuple[str, ...]]:
    corpus_edits: dict[tuple[int, int, int], tuple[str, ...]] = {}
    for sent_id, (source, target) in enumerate(zip(sources, targets)):
        for (i, j), replacement in extract_edits(source, target).items():
            corpus_edits[(sent_id, i, j)] = replacement
    return corpus_edits


def spellcheck_precision_recall(
    sources: Iterable[str],
    corrections: Iterable[str],
    predictions: Iterable[str],
) -> dict[str, float]:
    sources = list(sources)
    corrections = list(corrections)
    predictions = list(predictions)
    if not (len(sources) == len(corrections) == len(predictions)):
        raise ValueError("sources, corrections и predictions должны быть одной длины")

    gold_edits = extract_edits_corpus(sources, corrections)
    pred_edits = extract_edits_corpus(sources, predictions)
    tp = sum(1 for key, pred_val in pred_edits.items() if gold_edits.get(key) == pred_val)

    n_pred = len(pred_edits)
    n_gold = len(gold_edits)
    precision = tp / n_pred if n_pred else 1.0
    recall = tp / n_gold if n_gold else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "TP": tp,
        "num_gold_edits": n_gold,
        "num_pred_edits": n_pred,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1": round(f1 * 100, 2),
    }


def spellcheck_metrics_ruspelleval(
    sources: Iterable[str],
    corrections: Iterable[str],
    predictions: Iterable[str],
) -> dict[str, float]:
    sage_root = Path("sage")
    if sage_root.is_dir() and str(sage_root.resolve().parent) not in sys.path:
        sys.path.insert(0, str(sage_root.resolve().parent))

    from sage.evaluation import Scorer

    scorer = Scorer()
    return scorer.score(sources, corrections, predictions, metrics=["ruspelleval", "errant"])


## 5. Базовая модель SAGE-FREDT5-Large

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SAGE_MODEL = "ai-forever/sage-fredt5-large"

tokenizer = AutoTokenizer.from_pretrained(SAGE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(SAGE_MODEL).to("cuda")


In [ ]:
def predict_sage(sentences: list[str]) -> list[str]:
    predictions = []
    for sentence in sentences:
        inputs = tokenizer(
            sentence,
            max_length=None,
            padding="longest",
            truncation=False,
            return_tensors="pt",
        )
        outputs = model.generate(
            **inputs.to(model.device),
            max_length=int(inputs["input_ids"].size(1) * 1.5),
        )
        predictions.append(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])
    return predictions


demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print(predict_sage([demo_sentence])[0])


### 5.1 SAGE на подмножестве literature (MultidomainGold)

In [ ]:
pred_sage_literature = predict_sage(literature_with_errors["source"].tolist())
print(f"Predictions: {len(pred_sage_literature)}")


In [ ]:
print("RUSpellEval / ERRANT (SAGE, literature):")
print(spellcheck_metrics_ruspelleval(
    literature_with_errors["source"],
    literature_with_errors["correction"],
    pred_sage_literature,
))

print("\nEdit-level metrics (SAGE, literature):")
print(spellcheck_precision_recall(
    literature_with_errors["source"],
    literature_with_errors["correction"],
    pred_sage_literature,
))


### 5.2 SAGE на RU_SPELLCHECK_DEVICE (test)

In [ ]:
pred_sage = predict_sage(sources)
print(f"Predictions: {len(pred_sage)}")


In [ ]:
print("RUSpellEval / ERRANT (SAGE, RU_SPELLCHECK_DEVICE test):")
print(spellcheck_metrics_ruspelleval(sources, corrections, pred_sage))


## 6. Базовая модель Qwen3.5-0.8B

In [ ]:
import gc

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("VRAM освобождена после SAGE")


In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

import torch
torch._dynamo.config.recompile_limit = 64


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

QWEN_MODEL = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024

qwen_model, qwen_tokenizer = FastModel.from_pretrained(
    model_name=QWEN_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)

qwen_tokenizer = get_chat_template(
    qwen_tokenizer,
    chat_template="qwen3-instruct",
)


In [ ]:
def predict_qwen(sentences: list[str]) -> list[str]:
    predictions = []
    for sentence in sentences:
        messages = [
            {"role": "user", "content": f"Исправь ошибки в тексте: {sentence}"},
        ]
        text = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = qwen_tokenizer(
            text=[text],
            images=None,
            videos=None,
            return_tensors="pt",
        ).to(qwen_model.device)

        input_len = inputs["input_ids"].shape[1]
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max(int(len(sentence) * 1.5), 32),
            do_sample=False,
        )
        pred = qwen_tokenizer.decode(
            outputs[0][input_len:],
            skip_special_tokens=True,
        ).strip()
        predictions.append(pred)
    return predictions


demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print(predict_qwen([demo_sentence])[0])


In [ ]:
pred_qwen_base = predict_qwen(sources)
print(f"Predictions: {len(pred_qwen_base)}")


In [ ]:
print("RUSpellEval / ERRANT (Qwen3.5 base, RU_SPELLCHECK_DEVICE test):")
print(spellcheck_metrics_ruspelleval(sources, corrections, pred_qwen_base))


## 7. Дообученный Qwen3.5-0.8B

In [ ]:
import pandas as pd

QWEN_PREDICTIONS_PATH = "fine_result.csv"
pred_qwen = pd.read_csv(QWEN_PREDICTIONS_PATH)["prediction"].tolist()

print(f"Loaded {len(pred_qwen)} predictions from {QWEN_PREDICTIONS_PATH}")


In [ ]:
print("RUSpellEval / ERRANT (Qwen3.5 fine-tuned, RU_SPELLCHECK_DEVICE test):")
print(spellcheck_metrics_ruspelleval(sources, corrections, pred_qwen))
